<a href="https://colab.research.google.com/github/MaSBroEarl/-.-Text-classification/blob/main/DL%2BclassicML_%D0%BF%D0%BE%D0%B4%D1%85%D0%BE%D0%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Импортируем библиотеки

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

Загружаем данные

In [ ]:
df = pd.read_csv('/content/train.csv')

Препроцессинг

In [ ]:
# Предобработка
df = df.drop(['id', 'location'], axis=1)
df['keyword'] = df['keyword'].fillna('')

In [ ]:
# Делаем минимальную очистку текста,
# т.к трансформер хорошо работает с сырым текстом
def basic_clean(text):
    text = str(text).lower()
    text = text.replace('\n', ' ').strip()
    return text
df['clean_text'] = df['text'].apply(basic_clean)

In [ ]:
# Объединяем с keyword для лучшего контекста
df['text_for_model'] = df['keyword'] + ' ' + df['clean_text']

In [ ]:
print(f"После предобработки: {df.shape}")

Выбираем Эмбеддер all-MiniLM-L6-v2 (жертвуем качеством,так как нам важна скорость и небольшой размер моедели)

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'

In [ ]:
start_time = time.time()

In [ ]:
model = SentenceTransformer(model_name)
embedding_dim = model.get_sentence_embedding_dimension()

print(f"✅ Модель загружена за {time.time() - start_time:.1f} секунд")
print(f"Размерность эмбеддингов: {embedding_dim}")

Получаем эмбеддинги из предобученной модели

In [ ]:
# Разделяем данные
X = df['text_for_model'].values
y = df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {len(X_train)} записей")
print(f"Test: {len(X_test)} записей")

In [ ]:
# ГЕНЕРАЦИЯ ЭМБЕДДИНГОВ ДЛЯ TRAIN
tart_time = time.time()

# Обрабатываем батчами для экономии памяти
batch_size = 64
train_embeddings = []

for i in range(0, len(X_train), batch_size):
    batch = X_train[i:i+batch_size]
    batch_embeddings = model.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    train_embeddings.append(batch_embeddings)

train_embeddings = np.vstack(train_embeddings)
print(f"✅ Train эмбеддинги: {train_embeddings.shape}")
print(f"⏱ Время: {time.time() - start_time:.1f} сек")

In [ ]:
# ГЕНЕРАЦИЯ ЭМБЕДДИНГОВ ДЛЯ TEST
# ============================================
print("\n⏳ Генерация эмбеддингов для test...")
start_time = time.time()

test_embeddings = []

for i in range(0, len(X_test), batch_size):
    batch = X_test[i:i+batch_size]
    batch_embeddings = model.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    test_embeddings.append(batch_embeddings)

test_embeddings = np.vstack(test_embeddings)
print(f"✅ Test эмбеддинги: {test_embeddings.shape}")
print(f"⏱ Время: {time.time() - start_time:.1f} сек")

In [ ]:
# Сохраняем эмбеддинги
np.save('train_embeddings.npy', train_embeddings)
np.save('test_embeddings.npy', test_embeddings)
print("✅ Эмбеддинги сохранены на диск")

Берем LogisticRegression

In [ ]:
#Обучаем модель Logistic Regression на эмбеддингах
lr_emb = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0,
    class_weight='balanced'
)

start_time = time.time()
lr_emb.fit(train_embeddings, y_train)
print(f"✅ Модель обучена за {time.time() - start_time:.1f} сек")

In [ ]:
# ПРЕДСКАЗАНИЯ
y_train_pred_emb = lr_emb.predict(train_embeddings)
y_test_pred_emb = lr_emb.predict(test_embeddings)
y_test_proba_emb = lr_emb.predict_proba(test_embeddings)[:, 1]

Подсчет метрик

In [ ]:
train_acc_emb = accuracy_score(y_train, y_train_pred_emb)
train_prec_emb = precision_score(y_train, y_train_pred_emb)
train_rec_emb = recall_score(y_train, y_train_pred_emb)
train_f1_emb = f1_score(y_train, y_train_pred_emb)

test_acc_emb = accuracy_score(y_test, y_test_pred_emb)
test_prec_emb = precision_score(y_test, y_test_pred_emb)
test_rec_emb = recall_score(y_test, y_test_pred_emb)
test_f1_emb = f1_score(y_test, y_test_pred_emb)
test_roc_auc_emb = roc_auc_score(y_test, y_test_proba_emb)

In [ ]:
print("📊 Метрики на TRAIN:")
print(f"  Accuracy:  {train_acc_emb:.4f}")
print(f"  Precision: {train_prec_emb:.4f}")
print(f"  Recall:    {train_rec_emb:.4f}")
print(f"  F1-score:  {train_f1_emb:.4f}")

print("\n📊 Метрики на TEST:")
print(f"  Accuracy:  {test_acc_emb:.4f}")
print(f"  Precision: {test_prec_emb:.4f}")
print(f"  Recall:    {test_rec_emb:.4f}")
print(f"  F1-score:  {test_f1_emb:.4f}")
print(f"  ROC-AUC:   {test_roc_auc_emb:.4f}")

print(f"\n📊 Разница train-test: {train_acc_emb - test_acc_emb:.4f}")

In [ ]:
print("""
📊 ИТОГОВЫЙ РЕЙТИНГ МОДЕЛЕЙ:

┌─────────────────────────┬──────────┬──────────┬──────────┐
│ Модель                  │ Accuracy │ F1-score │ ROC-AUC  │
├─────────────────────────┼──────────┼──────────┼──────────┤
│ 1. Эмбеддинги + LR      │ 0.8148 ✅ │ 0.7883 ✅ │ 0.8889 ✅ │
│ 2. TF-IDF + LR          │ 0.8048   │ 0.7716   │ 0.8623   │
│ 3. XGBoost              │ 0.7548   │ 0.6364   │ 0.8269   │
└─────────────────────────┴──────────┴──────────┴──────────┘""")

Вывод: Наилучший результат показала модель логистической регрессии + эмбеддинги из предобученной модели all-MiniLM-L6-v2